#Setup

In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:
!nvidia-smi

Fri Mar 13 00:27:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpcuome5um".


# Parallel Vector Addition

In [ ]:
%%writefile parallel_vec_add.cu
#include<iostream>
#include<cmath>
#include<chrono>
#include<cuda_runtime.h>

__global__ void vec_add(const float* A, const float* B, float* C, int N){
  int i = blockDim.x * blockIdx.x + threadIdx.x;

  if (i < N){
    C[i] = A[i] + B[i];
  }
}

void run_vec_add(int N){
  //int N = 1 << 22;

  // Initialize arrays A, B, C on CPU
  //float A[N], B[N], C[N], C_CPU[N];
  float* A     = new float[N];
  float* B     = new float[N];
  float* C     = new float[N];
  float* C_CPU = new float[N];

  for(int i = 0; i < N ; i++){
    A[i] = (float)i + 1.0f;
    B[i] = 2.0f;
  }

  auto cpu_start = std::chrono::high_resolution_clock::now();
  for (int i = 0 ; i < N ; i++){
    C_CPU[i] = A[i] + B[i];
  }
  auto cpu_end = std::chrono::high_resolution_clock::now();
  std::chrono::duration<double, std::milli> cpu_time = cpu_end - cpu_start;
  std::cout << "CPU time (ms): " << cpu_time.count() << std::endl;

  // memory allocations A, B, C on GPU
  float *d_a, *d_b, *d_c;
  cudaMalloc(&d_a, N*sizeof(float));
  cudaMalloc(&d_b, N*sizeof(float));
  cudaMalloc(&d_c, N*sizeof(float));

  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);

  // copy the contents of the arrays on CPU to allocated memory on the GPU
  cudaMemcpy(d_a, A, N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, B, N*sizeof(float), cudaMemcpyHostToDevice);

  int blocksize = 256;
  int gridsize = (int)ceil((float)N/blocksize); //1
  vec_add<<<gridsize, blocksize>>>(d_a, d_b, d_c, N);

  cudaMemcpy(C, d_c, N*sizeof(float), cudaMemcpyDeviceToHost);

  cudaEventRecord(stop);
  cudaEventSynchronize(stop);

  float gpu_total_time = 0.0f;
  cudaEventElapsedTime(&gpu_total_time, start, stop);

  std::cout << "GPU total time (ms): " << gpu_total_time << std::endl;

  /*std::cout << "Vec A: " << std::endl;
  for(int i = 0 ; i<N ; i++){
    std::cout << A[i] << " ";
  }
  std::cout << "\n";

  std::cout << "Vec B: " << std::endl;
  for(int i = 0 ; i<N ; i++){
    std::cout << B[i] << " ";
  }
  std::cout << "\n";


  std::cout << "Vec C (GPU): " << std::endl;
  for(int i = 0 ; i<N ; i++){
    std::cout << C[i] << " ";
  }
  std::cout << "\n";

  std::cout << "Vec C (CPU): " << std::endl;
  for(int i = 0 ; i<N ; i++){
    std::cout << C_CPU[i] << " ";
  }
  std::cout << "\n";*/


  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);
  delete[] A;
  delete[] B;
  delete[] C;
  delete[] C_CPU;
}


int main() {
    // Define multiple vector sizes to compare
    int sizes[] = { 1024, 1 << 20, 1 << 22, 1 << 23 }; // 1K, 1M, 4M, 8M
    int num_sizes = sizeof(sizes) / sizeof(sizes[0]);

    std::cout << "CPU vs GPU vector addition times:\n";
    std::cout << "---------------------------------\n";

    for (int i = 0; i < num_sizes; i++) {
        int N = sizes[i];
        std::cout << "\nVector size: " << N << "\n";
        run_vec_add(N);
    }

    return 0;
}



Overwriting parallel_vec_add.cu


In [ ]:
!nvcc -arch=sm_75 parallel_vec_add.cu -o parallel_vec_add

In [ ]:
!./parallel_vec_add

CPU vs GPU vector addition times:
---------------------------------

Vector size: 1024
CPU time (ms): 0.003699
GPU total time (ms): 0.15792

Vector size: 1048576
CPU time (ms): 4.65388
GPU total time (ms): 5.20864

Vector size: 4194304
CPU time (ms): 18.702
GPU total time (ms): 17.8302

Vector size: 8388608
CPU time (ms): 37.3851
GPU total time (ms): 35.9648


#Matrix Addition

In [ ]:
%%writefile matrix_addition.cu
#include <iostream>
#include <cmath>

__global__ void matrix_add(const float* A, const float* B, float* C, int rows, int cols){
  int i = blockIdx.x * blockDim.x + threadIdx.x;
  int j = blockIdx.y * blockDim.y + threadIdx.y;
  if (i < rows && j < cols){
    int index = i * cols + j;
    C[index] = A[index] + B[index];
  }
}

int main(){
  int rows = 3;
  int cols = 3;

  int N = rows * cols;

  float* A = new float[N];
  float* B = new float[N];
  float* C = new float[N];

  for (int i = 0 ; i < N ; i++){
    A[i] = (float)i + 1.0f;
    B[i] = 2.0f;
    C[i] = 0.0f;
  }

  float *d_a, *d_b, *d_c;
  cudaMalloc(&d_a, N * sizeof(float));
  cudaMalloc(&d_b, N * sizeof(float));
  cudaMalloc(&d_c, N * sizeof(float));

  cudaMemcpy(d_a, A, N*sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_b, B, N*sizeof(float), cudaMemcpyHostToDevice);

  dim3 blockDim(32, 16);
  dim3 gridDim((int)ceil((float)N/blockDim.x),(int)ceil((float)N/blockDim.y));
  matrix_add<<<gridDim, blockDim>>>(d_a, d_b, d_c, rows, cols);
  cudaMemcpy(C, d_c, N*sizeof(float), cudaMemcpyDeviceToHost);

  std::cout << "A: \n";
  for (int i = 0 ; i < N ; i++){
    std::cout << A[i] << " ";
  }

  std::cout << "\nB: \n";
  for (int i = 0 ; i < N ; i++){
    std::cout << B[i] << " ";
  }

  std::cout << "\nC: \n";
  for (int i = 0 ; i < N ; i++){
    std::cout << C[i] << " ";
  }

  cudaFree(d_a);
  cudaFree(d_b);
  cudaFree(d_c);
  delete[] A;
  delete[] B;
  delete[] C;

}

Overwriting matrix_addition.cu


In [ ]:
!nvcc -arch=sm_75 matrix_addition.cu -o matrix_addition

In [ ]:
!./matrix_addition

A: 
1 2 3 4 5 6 7 8 9 
B: 
2 2 2 2 2 2 2 2 2 
C: 
3 4 5 6 7 8 9 10 11 

#Matrix-Vector Multiplication

In [ ]:
%%writefile multiplication.cu

#include<iostream>
#include<cmath>

__global__ void matrix_multiply(const float *A, const float *X, float *B, int A_ROWS, int A_COLS){
  int row = blockIdx.x * blockDim.x + threadIdx.x;
  if (row < A_ROWS){
    float sum = 0.0f;
    for (int j = 0 ; j < A_COLS ; j++){
      sum += A[row * A_COLS + j] * X[j];
    }
    B[row] = sum;
  }
}


int main(){
  int A_ROWS = 3;
  int A_COLS = 3;

  int X_ROWS = 3;
  int X_COLS = 1;

  float *A = new float[A_ROWS * A_COLS];
  float *X = new float[X_ROWS * X_COLS];
  float *B = new float[A_COLS * X_COLS];

  for (int i = 0 ; i < A_ROWS * A_COLS ; i++){
    A[i] = (float)i + 1.0f;
    if (i < X_ROWS * X_COLS){
      X[i] = 2.0f;
    }
  }

  for (int i = 0 ; i < A_COLS * X_COLS ; i++){
    B[i] = 0.0f;
  }

  float *d_a, *d_b, *d_x;

  cudaMalloc(&d_a, A_ROWS * A_COLS * sizeof(float));
  cudaMalloc(&d_x, X_ROWS * X_COLS * sizeof(float));
  cudaMalloc(&d_b, A_COLS * X_COLS * sizeof(float));

  cudaMemcpy(d_a, A, A_ROWS * A_COLS * sizeof(float), cudaMemcpyHostToDevice);
  cudaMemcpy(d_x, X, X_ROWS * X_COLS * sizeof(float), cudaMemcpyHostToDevice);

  int blocksize = 256;
  int gridsize = (int)ceil((float)A_ROWS/blocksize); //1
  matrix_multiply<<<gridsize, blocksize>>>(d_a, d_x, d_b, A_ROWS, A_COLS);
  cudaMemcpy(B, d_b, A_COLS * X_COLS * sizeof(float), cudaMemcpyDeviceToHost);

  for (int i = 0 ; i < A_COLS * X_COLS ; i++){
    std::cout << B[i] << " ";
  }
}

Overwriting multiplication.cu


In [ ]:
!nvcc -arch=sm_75 multiplication.cu -o multiplication

In [ ]:
!./multiplication

12 30 48 

# Blockwise prefix sum

In [ ]:
%%writefile blockwise_sum.cu

#include<iostream>

__global__ void blockwise_prefix(int *ip, int *op, int n){
  extern __shared__ int shared_mem[];

  int tid = threadIdx.x;
  int idx = blockIdx.x * blockDim.x * 2 + tid;

  if (idx < n){
    //blockdim.x = 8 threads in each block
    shared_mem[tid] = ip[idx] + ip[idx+blockDim.x]; //0+9, 1+10, 2+11,......
    __syncthreads();

    // result: [0+9, 1+10,.......,7+15] ---> 8 sized array from 16 size
    for (int stride = 1; stride < blockDim.x ; stride *= 2){
      int temp = 0;
      if (tid >= stride){
        temp = shared_mem[tid - stride];
      }
      __syncthreads();
      shared_mem[tid] += temp;
      __syncthreads();
    }

    op[idx] = shared_mem[tid];
  }
}


int main(){
  int N = 16;
  int input[16] = {1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16};
  int output[16] = {0};

  int *d_input, *d_output;

  cudaMalloc(&d_input, N * sizeof(int));
  cudaMalloc(&d_output, N * sizeof(int));

  cudaMemcpy(d_input, input, N * sizeof(int), cudaMemcpyHostToDevice);

  int blocksize = 8;
  int gridsize = N / blocksize;
  int shared_mem_size = blocksize * sizeof(int); //8 * 4 = 32 bytes of shared memory

  blockwise_prefix<<<gridsize, blocksize, shared_mem_size>>>(d_input, d_output, N);

  cudaMemcpy(output, d_output, N * sizeof(int), cudaMemcpyDeviceToHost);

  for (int i = 0 ; i < N ; i++){
    std::cout << output[i] << " ";
  }
}

Overwriting blockwise_sum.cu


In [ ]:
!nvcc -arch=sm_75 blockwise_sum.cu -o blockwise_sum

In [ ]:
!./blockwise_sum

10 22 36 52 70 90 112 136 0 0 0 0 0 0 0 0 

# Layer Norm

In [ ]:
%%writefile layer_norm.cu

#include<iostream>
#include<cmath>

__global__ void layer_norm(float *ip, float *op, int rows, int cols){

  int row = blockIdx.x;
  int col = threadIdx.x;
  extern __shared__ float shared_mem[];

  if (row < rows && col < cols){

    //each thread load the data
    float x = ip[row * cols + col];
    shared_mem[col] = x;
    __syncthreads();

    //each thread (col) inside a block (row) computes the mean for that row
    float mean = 0.0f;
    for (int i = 0 ; i < cols ; i++){
      mean += shared_mem[i];
    }
    mean /= cols;
    __syncthreads();

    // variance calculation
    float diff = x - mean;
    float var_term = diff*diff;
    shared_mem[col] = var_term;
    __syncthreads();

    float var = 0.0f;
    for (int i = 0 ; i < cols ; i++){
      var += shared_mem[i];
    }
    var /= cols;
    float inverse_std_dev = rsqrtf(var + 1e-7);

    op[row * cols + col] = diff * inverse_std_dev;
  }
}

int main(){
  int rows = 2;
  int cols = 4;

  float h_ip[rows * cols] = {
    1,2,3,4,
    2,4,6,8
  };

  float h_op[rows * cols] = {0};

  float *d_ip, *d_op;

  cudaMalloc(&d_ip, rows*cols*sizeof(float));
  cudaMalloc(&d_op, rows*cols*sizeof(float));

  cudaMemcpy(d_ip, h_ip, rows*cols*sizeof(float), cudaMemcpyHostToDevice);

  dim3 grid(rows);
  dim3 block(cols);

  size_t shared_mem_size = cols*sizeof(float);

  layer_norm<<<grid, block, shared_mem_size>>>(d_ip, d_op, rows, cols);
  cudaMemcpy(h_op, d_op, rows*cols*sizeof(float), cudaMemcpyDeviceToHost);

  std::cout << "CUDA LayerNorm kernel\n\n";
  std::cout << "Grid  size : " << grid.x << " blocks (1 block per row)\n";
  std::cout << "Block size : " << block.x << " threads (1 thread per column)\n";
  std::cout << "Shared mem : " << shared_mem_size << " bytes per block\n\n";

  std::cout << "Input Matrix:\n";
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            std::cout << h_ip[i * cols + j] << " ";
        }
        std::cout << "\n";
    }

  std::cout << "\n";

  std::cout << "LayerNorm Kernel Output:\n";
    for (int i = 0; i < rows; i++) {
        for (int j = 0; j < cols; j++) {
            std::cout << h_op[i * cols + j] << " ";
        }
        std::cout << "\n";
    }

    cudaFree(d_ip);
    cudaFree(d_op);

    return 0;
}

Overwriting layer_norm.cu


In [ ]:
!nvcc -arch=sm_75 layer_norm.cu -o layer_norm

In [ ]:
!./layer_norm

CUDA LayerNorm kernel

Grid  size : 2 blocks (1 block per row)
Block size : 4 threads (1 thread per column)
Shared mem : 16 bytes per block

Input Matrix:
1 2 3 4 
2 4 6 8 

LayerNorm Kernel Output:
-1.34164 -0.447214 0.447214 1.34164 
-1.34164 -0.447214 0.447214 1.34164 


#Matrix Transpose

In [ ]:
%%writefile matrix_transpose.cu

#include<stdio.h>

__global__ void matrix_transpose(float* ip, float* op, int ROWS, int COLS){
  int x = blockIdx.x * blockDim.x + threadIdx.x;
  int y = blockIdx.y * blockDim.y + threadIdx.y;

  if (x < ROWS && y < COLS){
    op[y * ROWS + x] = ip[x * COLS + y];
  }
}


int main(){
  int rows = 2;
  int cols = 3;

  float* h_ip = (float*)malloc(rows*cols*sizeof(float));
  float* h_op = (float*)malloc(cols*rows*sizeof(float));

  for (int i = 0 ; i < rows*cols ; i++){
    h_ip[i] = (float)i+1;
  }

  float *d_ip, *d_op;
  cudaMalloc(&d_ip, rows*cols*sizeof(float));
  cudaMalloc(&d_op, cols*rows*sizeof(float));

  cudaMemcpy(d_ip, h_ip, rows*cols*sizeof(float), cudaMemcpyHostToDevice);

  dim3 blocksize(32,32);
  dim3 gridsize((rows + blocksize.x - 1)/blocksize.x, (cols + blocksize.y - 1)/blocksize.y);

  matrix_transpose<<<gridsize, blocksize>>>(d_ip, d_op, rows, cols);
  cudaMemcpy(h_op, d_op, rows*cols*sizeof(float), cudaMemcpyDeviceToHost);

  printf("Input Matrix (%d x %d):\n", rows, cols);
  for (int i = 0; i < rows; i++) {
      for (int j = 0; j < cols; j++) {
          printf("%.1f  ", h_ip[i * cols + j]);
      }
      printf("\n");
  }

  printf("\nTransposed Matrix (%d x %d):\n", cols, rows);
  for (int i = 0; i < cols; i++) {
      for (int j = 0; j < rows; j++) {
          printf("%.1f  ", h_op[i * rows + j]);
      }
      printf("\n");
  }

  cudaFree(d_ip);
  cudaFree(d_op);
  free(h_ip);
  free(h_op);


}

Overwriting matrix_transpose.cu


In [ ]:
!nvcc -arch=sm_75 matrix_transpose.cu -o matrix_transpose

In [ ]:
!./matrix_transpose

Input Matrix (2 x 3):
1.0  2.0  3.0  
4.0  5.0  6.0  

Transposed Matrix (3 x 2):
1.0  4.0  
2.0  5.0  
3.0  6.0  


#Matrix Transpose (TILING)

In [ ]:
%%writefile matrix_transpose.cu

#include<stdio.h>
#define TILE_DIM 32

__global__ void matrix_transpose(float* ip, float* op, int ROWS, int COLS){
  __shared__ float tile[TILE_DIM][TILE_DIM+1];

  int x = blockIdx.x * TILE_DIM + threadIdx.x; //cols_idx
  int y = blockIdx.y * TILE_DIM + threadIdx.y; //rows_idx

  //writing to tile
  if (x < COLS && y < ROWS){
    tile[threadIdx.y][threadIdx.x] = ip[y * COLS + x]; //ip[row * COLS + col]
  }
  __syncthreads();

  x = blockIdx.y * TILE_DIM + threadIdx.x; //col in output
  y = blockIdx.x * TILE_DIM + threadIdx.y; //row in output

  //writing to global
  if (x < ROWS && y < COLS){
    op[y * ROWS + x] = tile[threadIdx.x][threadIdx.y];
  }
}


int main(){
  int rows = 2;
  int cols = 3;

  float* h_ip = (float*)malloc(rows*cols*sizeof(float));
  float* h_op = (float*)malloc(cols*rows*sizeof(float));

  for (int i = 0 ; i < rows*cols ; i++){
    h_ip[i] = (float)i+1;
  }

  float *d_ip, *d_op;
  cudaMalloc(&d_ip, rows*cols*sizeof(float));
  cudaMalloc(&d_op, cols*rows*sizeof(float));

  cudaMemcpy(d_ip, h_ip, rows*cols*sizeof(float), cudaMemcpyHostToDevice);

  dim3 blocksize(32,32);
  dim3 gridsize((cols + blocksize.x - 1)/blocksize.x, (rows + blocksize.y - 1)/blocksize.y);

  matrix_transpose<<<gridsize, blocksize>>>(d_ip, d_op, rows, cols);
  cudaMemcpy(h_op, d_op, rows*cols*sizeof(float), cudaMemcpyDeviceToHost);

  printf("Input Matrix (%d x %d):\n", rows, cols);
  for (int i = 0; i < rows; i++) {
      for (int j = 0; j < cols; j++) {
          printf("%.1f  ", h_ip[i * cols + j]);
      }
      printf("\n");
  }

  printf("\nTransposed Matrix (%d x %d):\n", cols, rows);
  for (int i = 0; i < cols; i++) {
      for (int j = 0; j < rows; j++) {
          printf("%.1f  ", h_op[i * rows + j]);
      }
      printf("\n");
  }

  cudaFree(d_ip);
  cudaFree(d_op);
  free(h_ip);
  free(h_op);


}

Overwriting matrix_transpose.cu


In [ ]:
!nvcc -arch=sm_75 matrix_transpose.cu -o matrix_transpose

In [ ]:
!./matrix_transpose

Input Matrix (2 x 3):
1.0  2.0  3.0  
4.0  5.0  6.0  

Transposed Matrix (3 x 2):
1.0  4.0  
2.0  5.0  
3.0  6.0  


#Neural Network

In [6]:
%%writefile neural_network.cu

#include <iostream>
#include <cuda_runtime.h>
#include <curand.h>
#include <cublas_v2.h>
#include <chrono>


__global__ void addBias(float *output, float *bias, int rows, int cols){
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  if (row < rows && col < cols)
        output[row * cols + col] += bias[col];
}

__global__ void applyReLU(float *input, float *output, int rows, int cols){
  int row = blockIdx.y * blockDim.y + threadIdx.y;
  int col = blockIdx.x * blockDim.x + threadIdx.x;
  if (row < rows && col < cols){
    output[row * cols + col] = fmaxf(input[row * cols + col], 0.0f);
  }
}

__global__ void applySoftmax(float *input, float *output, int rows, int cols){
  int row = blockIdx.x;
  int col = threadIdx.x;
  extern __shared__ float shared_mem[]; //each block has its own shared memory

  if (row < rows && col < cols){

    //each thread load the data
    float x = input[row * cols + col];
    shared_mem[col] = x;
    __syncthreads();

    //each thread finds the largest value in the row
    float max_val = shared_mem[0];
    for (int i = 0 ; i < cols ; i++){
      if (shared_mem[i] > max_val){
        max_val = shared_mem[i];
      }
    }
    __syncthreads();

    //each thread subtract max and exponentiate each element of the row
    shared_mem[col] = expf(shared_mem[col] - max_val);
    __syncthreads();

    //each thread has the sum of the row
    float sum = 0.0f;
    for (int i = 0 ; i < cols ; i++){
      sum += shared_mem[i];
    }
    __syncthreads();

    //each thread divides each element by the sum of the respective row
    output[row * cols + col] = shared_mem[col] / sum;
  }
}

__global__ void softmax_backprop(float *dinputs, float *dvalues, int *y_true, int rows, int cols) {
    int row = blockIdx.x;
    int col = threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;

        // copy dvalues into dinputs
        dinputs[idx] = dvalues[idx];

        // subtract 1 at true class index first
        if (col == y_true[row])
            dinputs[idx] -= 1.0f;

        // then divide by rows
        dinputs[idx] /= rows;
    }
}

__global__ void reluBackward(float *dinputs, float *dvalues, float *inputs, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        // copy dvalues, but zero out where original input was <= 0
        dinputs[idx] = inputs[idx] > 0.0f ? dvalues[idx] : 0.0f;
    }
}

__global__ void colSumKernel(float *dvalues, float *dbiases, int rows, int cols) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (col < cols) {
        float sum = 0.0f;
        for (int row = 0; row < rows; row++)
            sum += dvalues[row * cols + col];
        dbiases[col] = sum;
    }
}

__global__ void sgdUpdateKernel(float *params, float *dparams, float lr, int size) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < size)
        params[idx] -= lr * dparams[idx];
}

class LayerDense{
  public:
    float *inputs;
    float *weights;
    float *bias;
    float *output;

    float *dweights;
    float *dbiases;
    float *dinputs;

    int n_inputs, n_neurons, batch_size;
    cublasHandle_t handle;

    LayerDense(int n_inputs, int n_neurons, int batch_size = 1):n_inputs(n_inputs), n_neurons(n_neurons), batch_size(batch_size){
      dweights = nullptr;
      dbiases  = nullptr;
      dinputs  = nullptr;

      cublasCreate(&handle);

      //W [n_inputs X n_neurons]
      //X [batch_size X n_inputs]
      // output [batch_size X n_neurons] = X[batch_size X n_inputs] * W[n_inputs X n_neurons]

      cudaMalloc(&weights, n_inputs * n_neurons * sizeof(float));
      cudaMalloc(&bias, n_neurons * sizeof(float));
      cudaMalloc(&output, batch_size * n_neurons * sizeof(float));

      //weights matrix initialize
      curandGenerator_t gen;
      curandCreateGenerator(&gen, CURAND_RNG_PSEUDO_DEFAULT);
      curandSetPseudoRandomGeneratorSeed(gen, 1234ULL);
      curandGenerateNormal(gen, weights, n_inputs * n_neurons, 0.0f, 0.01f); // mean=0, std=0.01
      curandDestroyGenerator(gen);

      //bias initialization
      cudaMemset(bias, 0, n_neurons * sizeof(float));
    }

    void forward(float *input){
      inputs = input;
      const float alpha = 1.0f, beta = 0.0f;

      // weights^T  [n_neurons  x n_inputs  ]
      // input^T    [n_inputs   x batch_size]
      // output^T   [n_neurons  x batch_size]

      // We wanna do output = W^T * X^T
      cublasSgemm(handle,
            CUBLAS_OP_N, CUBLAS_OP_N, // no transpose
            n_neurons,  // rows of W^T
            batch_size, // colums of X^T
            n_inputs,   // common dimension
            &alpha,
            weights, n_neurons, //leading dimension of W^T
            input,   n_inputs, //leading dimension of X^T
            &beta,
            output,  n_neurons //leading dimension of output^T
        );

      dim3 blocksize(16, 16);
      dim3 gridsize((n_neurons  + blocksize.x - 1) / 16, //cols
                  (batch_size  + blocksize.y - 1) / 16); //rows
      addBias<<<gridsize, blocksize>>>(output, bias, batch_size, n_neurons);
    }

    void backward(float *dvalues) {
      if (!dweights) cudaMalloc(&dweights, n_inputs  * n_neurons  * sizeof(float));
      if (!dbiases) cudaMalloc(&dbiases,  n_neurons * sizeof(float));
      if (!dinputs) cudaMalloc(&dinputs,  batch_size * n_inputs  * sizeof(float));

      const float alpha = 1.0f, beta = 0.0f;

      // 1-- dweights = inputs^T * dvalues
      // inputs^T  [n_inputs  x batch_size]  → A in cuBLAS
      // dvalues   [batch_size x n_neurons]  → B in cuBLAS
      // dweights  [n_inputs  x n_neurons]   → C in cuBLAS
      // cuBLAS: m=n_neurons, n=n_inputs, k=batch_size
      cublasSgemm(handle,
          CUBLAS_OP_N, CUBLAS_OP_T,   // transpose inputs
          n_neurons,   // m
          n_inputs,    // n
          batch_size,  // k
          &alpha,
          dvalues, n_neurons,  // A = dvalues^T
          inputs,  n_inputs,   // B = inputs (op=T gives inputs^T)
          &beta,
          dweights, n_neurons  // C = dweights^T
      );

      // 2-- dbiases = colwise sum of dvalues
      colSumKernel<<<(n_neurons + 255) / 256, 256>>>(dvalues, dbiases, batch_size, n_neurons);

      // 3-- dinputs = dvalues * weights^T
      // dvalues  [batch_size x n_neurons]   → A in cuBLAS
      // weights^T[n_neurons  x n_inputs]    → B in cuBLAS
      // dinputs  [batch_size x n_inputs]    → C in cuBLAS
      // cuBLAS: m=n_inputs, n=batch_size, k=n_neurons
      cublasSgemm(handle,
          CUBLAS_OP_T, CUBLAS_OP_N,   // transpose weights
          n_inputs,    // m
          batch_size,  // n
          n_neurons,   // k
          &alpha,
          weights,  n_neurons,  // A = weights (op=T gives weights^T)
          dvalues,  n_neurons,  // B = dvalues^T
          &beta,
          dinputs,  n_inputs    // C = dinputs^T
      );
  }
};

class ReLU{
  public:
    float *inputs;
    float *outputs;
    float *dinputs;

    ReLU() : outputs(nullptr), dinputs(nullptr) {}

    void forward(float *input, int rows, int cols){
      inputs = input;
      if (!outputs) cudaMalloc(&outputs, rows * cols * sizeof(float));
      dim3 blocksize(16, 16);
      dim3 gridsize((cols  + blocksize.x - 1) / blocksize.x, //cols
                  (rows  + blocksize.y - 1) / blocksize.y); //rows
      applyReLU<<<gridsize, blocksize>>>(input, outputs, rows, cols);
    }

    void backward(float *dvalues, int rows, int cols){
      if (!dinputs) cudaMalloc(&dinputs, rows * cols * sizeof(float));

      dim3 blocksize(16, 16);
      dim3 gridsize((cols + blocksize.x - 1) / blocksize.x,
                  (rows + blocksize.y - 1) / blocksize.y);

      reluBackward<<<gridsize, blocksize>>>(dinputs, dvalues, inputs, rows, cols);
    }
};

class Softmax{
  public:
    float *outputs;
    float *dinputs;

    Softmax() : outputs(nullptr), dinputs(nullptr) {}

    double forward(float *input, int *y_true, int rows, int cols) {
        if (!outputs) cudaMalloc(&outputs, rows * cols * sizeof(float));
        applySoftmax<<<rows, cols, cols * sizeof(float)>>>(input, outputs, rows, cols);
        cudaDeviceSynchronize();

        // copy outputs back to host for loss computation
        float *h_outputs = new float[rows * cols];
        cudaMemcpy(h_outputs, outputs, rows * cols * sizeof(float), cudaMemcpyDeviceToHost);

        // cross entropy loss — only look at the probability of the true class
        double epsilon = 1e-7;
        double loss = 0.0;
        for (int i = 0; i < rows; i++) {
            int idx = y_true[i];                          // true class index for this sample
            double prob = h_outputs[i * cols + idx];      // probability at true class
            double clipped = std::max(epsilon, std::min(1.0 - epsilon, prob));
            loss += -log(clipped);
        }

        return loss / rows;
    }

    void backward(float *dvalues, int *y_true, int rows, int cols){
      if (!dinputs) cudaMalloc(&dinputs, rows * cols * sizeof(float));
      // one block per row, one thread per col — same as softmax forward
      softmax_backprop<<<rows, cols>>>(dinputs, dvalues, y_true, rows, cols);
    }
};

class SGD{
  public:
    SGD(float lr = 1.0f, float decay_value = 0.0f) : learning_rate(lr), current_lr(lr), decay(decay_value), iteration(0) {}

    void pre_update_params(){
      if (decay > 0){
            current_lr = learning_rate  * (1.0f / (1.0f + decay * iteration));
        }
    }

    void update_params(LayerDense &layer){
      int weight_size = layer.n_inputs  * layer.n_neurons;
      int bias_size   = layer.n_neurons;

      sgdUpdateKernel<<<(weight_size + 255) / 256, 256>>>(layer.weights, layer.dweights, current_lr, weight_size);

      sgdUpdateKernel<<<(bias_size + 255) / 256, 256>>>(layer.bias, layer.dbiases, current_lr, bias_size);
    }

    void post_update_params(){
      iteration += 1;
    }
  private:
    double learning_rate;
    double current_lr;
    double decay;
    std::size_t iteration;
};


int main() {
    int batch_size = 4;
    int n_inputs   = 2;
    int hidden     = 32;
    int n_classes  = 2;

    // XOR input [4x2]
    float h_input[] = {0,0, 0,1, 1,0, 1,1};
    int h_y_true[]  = {1,0,0,1};

    // copy input to device
    float *d_input;
    cudaMalloc(&d_input, batch_size * n_inputs * sizeof(float));
    cudaMemcpy(d_input, h_input, batch_size * n_inputs * sizeof(float), cudaMemcpyHostToDevice);

    int *d_y_true;
    cudaMalloc(&d_y_true, batch_size * sizeof(int));
    cudaMemcpy(d_y_true, h_y_true, batch_size * sizeof(int), cudaMemcpyHostToDevice);

    // build network
    LayerDense dense1(n_inputs,  hidden,     batch_size);
    ReLU       relu1;
    LayerDense dense2(hidden,    n_classes,  batch_size);
    Softmax    softmax;
    SGD        optimizer(0.1f);

    std::cout << "Starting training...\n";

    auto start = std::chrono::high_resolution_clock::now();

    // simple training loop
    for(int epoch = 0; epoch <= 10000; epoch++){
        // forward
        dense1.forward(d_input);
        relu1.forward(dense1.output, batch_size, hidden);
        dense2.forward(relu1.outputs);
        double loss = softmax.forward(dense2.output, h_y_true, batch_size, n_classes);

        // print loss every 1000 epochs
        if(epoch % 1000 == 0)
            std::cout << "Epoch " << epoch << " Loss: " << loss << "\n";

        // backward
        softmax.backward(softmax.outputs, d_y_true, batch_size, n_classes);
        dense2.backward(softmax.dinputs);
        relu1.backward(dense2.dinputs, batch_size, hidden);
        dense1.backward(relu1.dinputs);

        // update
        optimizer.pre_update_params();
        optimizer.update_params(dense1);
        optimizer.update_params(dense2);
        optimizer.post_update_params();
    }

    auto stop = std::chrono::high_resolution_clock::now();

    std::chrono::duration<double> elapsed = stop - start;
    std::cout << "\nTraining completed in: " << elapsed.count() << " seconds\n";

    // --- Inference ---
    std::cout << "\nXNOR Predictions after Training:\n";

    // One last forward pass
    dense1.forward(d_input);
    relu1.forward(dense1.output, batch_size, hidden);
    dense2.forward(relu1.outputs);
    softmax.forward(dense2.output, h_y_true, batch_size, n_classes); // Loss return ignored

    // Copy the final probabilities from device to host
    float *h_final_probs = new float[batch_size * n_classes];
    cudaMemcpy(h_final_probs, softmax.outputs, batch_size * n_classes * sizeof(float), cudaMemcpyDeviceToHost);

    for (int i = 0; i < batch_size; i++) {
        float prob0 = h_final_probs[i * n_classes + 0];
        float prob1 = h_final_probs[i * n_classes + 1];

        // Determine predicted class (0 or 1)
        int prediction = (prob1 > prob0) ? 1 : 0;

        std::cout << "Input: (" << h_input[i*2] << ", " << h_input[i*2+1]
                  << ") | True: " << h_y_true[i]
                  << " | Pred: " << prediction
                  << " (Prob: " << (prediction == 1 ? prob1 : prob0) << ")\n";
    }



    delete[] h_final_probs;

    cudaFree(d_input);
    cudaFree(d_y_true);

    return 0;
}

Overwriting neural_network.cu


In [7]:
!nvcc -arch=sm_75 neural_network.cu -o neural_network -lcurand -lcublas

In [8]:
!./neural_network

Starting training...
Epoch 0 Loss: 0.693163
Epoch 1000 Loss: 0.0190698
Epoch 2000 Loss: 0.00497351
Epoch 3000 Loss: 0.00260601
Epoch 4000 Loss: 0.00170465
Epoch 5000 Loss: 0.00124403
Epoch 6000 Loss: 0.000968413
Epoch 7000 Loss: 0.000786995
Epoch 8000 Loss: 0.000659366
Epoch 9000 Loss: 0.000565085
Epoch 10000 Loss: 0.000492884

Training completed in: 0.952798 seconds

XNOR Predictions after Training:
Input: (0, 0) | True: 1 | Pred: 1 (Prob: 0.999015)
Input: (0, 1) | True: 0 | Pred: 0 (Prob: 0.999663)
Input: (1, 0) | True: 0 | Pred: 0 (Prob: 0.999662)
Input: (1, 1) | True: 1 | Pred: 1 (Prob: 0.99969)
